<img src="logo.png" alt="Vegeta" width="240">

# Fixed-wing drone — durability (09b)

The second of two notebooks for one machine: how long the airframe of **`09a_fixed_wing_design`** lasts.
**Part 1** takes the preferred wing through vibration and fatigue (modes with the nacelles on the wing, the
Campbell diagram against the propeller lines, three missions, rainflow spectra, damage, the resonance amplitude,
a design comparison). **Part 2** does the same for the whole fuselage with its tail over a long full-battery
mission (modes, unit cases, gusts, spectrum, damage and life).

It starts from 09a's hand-off file (mass budget, preferred wing, propeller operating points). Without it, the
values recorded from a run of 09a are used, so this notebook also runs on its own.

```
Part 1  wing life:      modes + Campbell → unit stress fields → missions → spectra → damage → resonance amplitude → design comparison
Part 2  fuselage life:  regions + masses → modes → unit cases → long gust mission → spectrum → damage → life
```

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, aeromant, mellonia, core, boreas, chronos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
from vegeta.dedalus.examples import Propeller as PropellerCAD

ROOT = Path("_runs/fixed_wing_durability"); shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
MOVIES = Path("output") / "09b_fixed_wing_durability"; MOVIES.mkdir(parents=True, exist_ok=True)   # every movie of this notebook
RHO, G = 1.2, 9.81                      # air density [kg/m^3], gravity [m/s^2]
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"

## Hand-off from 09a

The design notebook writes these numbers at its end. The propeller and motor are rebuilt here from the same
definitions as in 09a; the operating points (rpm, thrust) come from the hand-off.

In [ ]:
from types import SimpleNamespace

RECORDED = {  # from a run of 09a (VEGETA_SKIP_OPENFOAM=1, recorded polar point); used when the hand-off file is missing
    "source": "recorded", "auw_kg": 0.9809, "wing_area_m2": 0.17, "v_cruise_m_s": 14.0, "rho": 1.2, "endurance_min": 67.6,
    "preferred_wing": {"revision": None, "thickness": 0.15},
    "parts_g": {"motors 2212 (2x)": 110.0, "propellers 9x6 (2x)": 24.0, "ESC 30 A (2x)": 50.0, "battery 3S 5000 mAh": 380.0,
                "flight controller + GPS": 35.0, "servos (4x)": 48.0, "receiver, wiring, bolts": 45.0, "payload (camera)": 150.0,
                "airframe (shells, from CAD area)": 138.9},
    "points": {"cruise": {"rpm": 5799.7, "thrust_N": 0.748}, "climb": {"rpm": 8940.5, "thrust_N": 5.087},
               "engine_out": {"rpm": 8998.2, "thrust_N": 4.567}},
}
HANDOFF = Path("_runs/fixed_wing_handoff.json")
H = json.loads(HANDOFF.read_text()) if HANDOFF.is_file() else RECORDED
print(f"hand-off: {'from ' + H['source'] if HANDOFF.is_file() else 'recorded values (run 09a to refresh them)'}")

AUW_kg, S, V_CRUISE, endurance_min = H["auw_kg"], H["wing_area_m2"], H["v_cruise_m_s"], H["endurance_min"]
RHO = H["rho"]
parts = pd.DataFrame({"mass_g": H["parts_g"]}).rename_axis("part")
preferred_wing = SimpleNamespace(params={"thickness": H["preferred_wing"]["thickness"]}, id=H["preferred_wing"]["revision"])
cruise, climb, one_engine = (SimpleNamespace(rpm=H["points"][k]["rpm"], thrust=H["points"][k]["thrust_N"]) for k in ("cruise", "climb", "engine_out"))

drone = dedalus.load_design("designs/fixed_wing.py:FixedWing")
LW_PLA = talos.Material("LW-PLA printed wing (solid-equivalent)", youngs_modulus=400.0, poissons_ratio=0.35,
                        density=0.6e-9, yield_strength=12.0, source="assumed; replace with coupon tests")
d, p = boreas.inches(9, 6)                           # the propeller and motor of 09a, same definitions
prop = boreas.Propeller.from_pitch("9x6 electric", d, p, blades=2, chord_root_m=0.014, chord_max_m=0.022, chord_tip_m=0.006,
                                   mass_kg=0.012, rotor_mass_kg=0.045, notes="generic planform")
motor = boreas.Motor("2212-920KV", kv_rpm_per_volt=920, resistance_ohm=0.12, no_load_current_a=0.6, max_current_a=20, mass_kg=0.055)
pd.DataFrame(H["points"]).T.round(2)

# Part 1 — the wing: vibration, cyclic loads and life

In [ ]:
RUNS = ROOT / "life"; RUNS.mkdir()
PREFERRED = dict(thickness=preferred_wing.params["thickness"])       # 09a: the preferred wing revision
AUW_KG, WING_AREA_M2 = AUW_kg, S
PROP = {name: {"rpm": pt.rpm, "thrust_N": pt.thrust, "unbalance_N": boreas.excitations(prop, pt.rpm)["unbalance_force_n"]}
        for name, pt in (("cruise", cruise), ("climb", climb), ("engine_out", one_engine))}       # 09a operating points
BLADES = prop.blades
NACELLE_MASS_T = (motor.mass_kg + prop.mass_kg + 0.025) * 1e-3     # motor + prop + ESC per nacelle, tonnes
W = AUW_KG * 9.81
LIFT_UNIT_MPA = W / (WING_AREA_M2 * 1e6)                      # pressure for n = 1 (level flight) in MPa
print(f"preferred wing {PREFERRED} | nacelle mass {NACELLE_MASS_T * 1e6:.0f} g")
pd.DataFrame(PROP).round(2)

## 1. Wing, masses, one mesh

In [ ]:
p = drone.resolve(part="wing", **PREFERRED)
wing = drone.generate(part="wing", **PREFERRED)
cad = wing.export(RUNS / "cad")             # LW_PLA: the wing material of 09a

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag for right in (False, True)]

yc, ny, d, f = p["fuselage_diameter"] / 2 + 5.0, p["nacelle_y"], p["nacelle_diameter"] / 2 + 1, p["nacelle_forward"]
REGIONS = [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)),
           talos.Surfaces("lift", lower_skins(cad.artifacts["step"])),
           talos.SurfacesInBox("motor_left", (-f - 0.1, -ny - d, -d, -f + 0.1, -ny + d, d)),
           talos.SurfacesInBox("motor_right", (-f - 0.1, ny - d, -d, -f + 0.1, ny + d, d))]
MASSES = [talos.PointMass("motor_left", NACELLE_MASS_T), talos.PointMass("motor_right", NACELLE_MASS_T)]
MESH = talos.MeshSettings(element_size=12.0)

def model(loads, name, step=None):
    return talos.StructuralModel(step or cad.artifacts["step"], "mm-N-MPa", LW_PLA, REGIONS, [talos.FixedSupport("root")],
                                 loads, MESH, name=name, masses=MASSES)

base = model([], "modal")
print(base.mesh(RUNS / "mesh", progress=True))

def case_dir(name):
    d_ = RUNS / name
    if not d_.exists():
        shutil.copytree(RUNS / "mesh", d_)
    return d_

## 2. Modes with the nacelles on the wing, and the Campbell diagram

The frequency diagram: 1P and blade-pass lines against rpm, the wing modes as horizontal lines, the operating points marked.

In [ ]:
modes = base.solve_modes(case_dir("modal"), n_modes=8, progress=True)
freqs = modes.metrics["frequencies_hz"]
print([round(x, 1) for x in freqs])
tviz.show(tviz.plot_mode(modes, mode=1))     # first wing bending

In [ ]:
tviz.show(tviz.plot_mode(modes, mode=7))     # the mode near the cruise shaft frequency

In [ ]:
structure = chronos.Structure(tuple(freqs), damping_ratio=0.03, source="Talos modal, LW-PLA solid-equivalent E, nacelle masses")
fig = structure.campbell({"1P": 1, "2P": BLADES}, np.linspace(2000, 10000, 30), operating_rpm={k: v["rpm"] for k, v in PROP.items()})
lines = {k: v["rpm"] / 60 for k, v in PROP.items()}
margins = pd.DataFrame({k: {"1P_hz": fq, "nearest_mode_hz": structure.nearest_mode(fq), "margin": structure.margin(fq),
                            "amplification": float(structure.amplification(fq)[0])} for k, fq in lines.items()}).round(2)
margins

### A finding, and a decision

The cruise shaft frequency (≈ 99 Hz) falls within a few percent of a wing mode: the dynamic
amplification is an order of magnitude, the usual comfort margin is 20 %. Three ways out, all
engineer decisions, none of them a change the software makes on its own:

1. **move the excitation** — cruise at a different rpm (a different propeller pitch, notebook 12);
2. **move the mode** — a stiffer or lighter nacelle mount, a spar;
3. **accept and verify** — run the fatigue with the amplification and see whether the life is still acceptable.

This notebook does (3) with the amplified vibration in the spectrum, and at the end repeats the
assessment for the thinner NACA 2412 wing to show how far a design change moves the numbers.

## 3. Unit load cases

| pattern | unit case | level unit |
|---|---|---|
| `lift` | pressure for n = 1 on the outer-panel lower skins | load factor n |
| `thrust` | +6 N on both nacelle noses | N per motor |
| `thrust_left` | +8 N on the left nacelle only (engine-out) | N |
| `vib_left` | 1 N vertical at the left nacelle nose (rotor unbalance) | N |

In [ ]:
UNIT = {
    "lift":        (model([talos.Pressure("lift", LIFT_UNIT_MPA)], "lift"), 1.0),
    "thrust":      (model([talos.Force("motor_left", fx=6.0), talos.Force("motor_right", fx=6.0)], "thrust"), 6.0),
    "thrust_left": (model([talos.Force("motor_left", fx=8.0)], "thrust_left"), 8.0),
    "vib_left":    (model([talos.Force("motor_left", fz=1.0)], "vib_left"), 1.0),
}
unit_results = {}
for name, (m, load) in tqdm(UNIT.items(), desc="unit cases"):
    unit_results[name] = m.solve(case_dir(name))
    r = unit_results[name]
    print(f"{name:<12} {'ok' if r.ok else 'FAILED'}  max von Mises {r.metrics.get('max_von_mises', float('nan')):.3f} MPa at level {load:g}")
unit_cases = {k: (unit_results[k], UNIT[k][1]) for k in UNIT}
tviz.show(tviz.plot_results(unit_results["vib_left"], field="von_mises"))

## 4. Three missions

`lift` levels are load factors (1 = level flight, 1.4 = a 45° banked turn, 2.5 = the pull-up of
notebook 09); gusts are `repeat`ed excursions. Vibration: unbalance on the left nacelle at the shaft
frequency of the segment's rpm (right nacelle assumed identical by symmetry — the spectrum counts one
side; the hotspot map shows where).

In [ ]:
def unb(point):
    return chronos.Excitation(f"unbalance {point}", PROP[point]["rpm"] / 60, PROP[point]["unbalance_N"], "vib_left")

TC, TCL = PROP["cruise"]["thrust_N"], PROP["climb"]["thrust_N"]
missions = {
    "survey": chronos.Mission("survey", (
        chronos.Segment("take-off + climb", 60, {"lift": 1.2, "thrust": TCL}, (unb("climb"),)),
        chronos.Segment("mapping legs", 2400, {"lift": 1.0, "thrust": TC}, (unb("cruise"),)),
        chronos.Segment("turn between legs", 8, {"lift": 1.3, "thrust": TC}, (unb("cruise"),), repeat=16),
        chronos.Segment("light turbulence", 2, {"lift": 1.25, "thrust": TC}, (unb("cruise"),), repeat=80),
        chronos.Segment("descent + landing", 90, {"lift": 0.9, "thrust": 0.3}),
        chronos.Segment("touchdown", 1, {"lift": 1.8}),
    ), "45 min mapping survey: long straight legs, gentle turns"),
    "patrol": chronos.Mission("patrol", (
        chronos.Segment("take-off + climb", 60, {"lift": 1.2, "thrust": TCL}, (unb("climb"),)),
        chronos.Segment("loiter", 1500, {"lift": 1.15, "thrust": TC}, (unb("cruise"),)),
        chronos.Segment("steep turn", 6, {"lift": 1.6, "thrust": TC}, (unb("cruise"),), repeat=60),
        chronos.Segment("evasive pull-up", 2, {"lift": 2.5, "thrust": TCL}, (unb("climb"),), repeat=4),
        chronos.Segment("engine-out drill", 30, {"lift": 1.0, "thrust_left": PROP["engine_out"]["thrust_N"]}, (unb("engine_out"),)),
        chronos.Segment("descent + landing", 90, {"lift": 0.9, "thrust": 0.3}),
        chronos.Segment("touchdown", 1, {"lift": 2.0}),
    ), "30 min patrol: continuous loiter turns, a few hard pull-ups, one engine-out drill"),
    "windy_hops": chronos.Mission("windy_hops", (
        chronos.Segment("take-off + climb", 45, {"lift": 1.3, "thrust": TCL}, (unb("climb"),), repeat=6),
        chronos.Segment("short cruise", 300, {"lift": 1.0, "thrust": TC}, (unb("cruise"),), repeat=6),
        chronos.Segment("gust", 1.5, {"lift": 1.7, "thrust": TC}, (unb("cruise"),), repeat=300),
        chronos.Segment("hard touchdown", 1, {"lift": 2.5}, repeat=6),
    ), "six short hops in gusty wind: 300 gusts, six hard touchdowns, ~35 min"),
}
for m in missions.values():
    fig = m.profile(patterns=["lift", "thrust", "thrust_left"])
pd.DataFrame({k: {"duration_min": m.duration_h * 60, "segments": len(m.segments)} for k, m in missions.items()}).T

In [ ]:
spectra = {k: chronos.build_spectrum(m, structure) for k, m in missions.items()}
for k, sp in spectra.items():
    sp.save(RUNS / f"spectrum_{k}.json")
    fig = sp.plot()
spectra["patrol"].table().round(4)

## 5. Damage per mission, hotspot, static re-check

S-N for LW-PLA — **assumed**: σ_f = 22 MPa, b = −0.12, Goodman with 14 MPa ultimate. Printed foamed
PLA is weak in fatigue and sensitive to temperature; coupon tests first.

In [ ]:
CURVE = talos.FatigueCurve("LW-PLA (assumed)", sigma_f=22.0, b=-0.12, ultimate=14.0, source="assumed; coupon tests needed")
fatigue = {k: talos.assess_fatigue(unit_cases, spectra[k].to_dict(), CURVE, workdir=RUNS / f"fatigue_{k}") for k in tqdm(missions, desc="fatigue")}
life = pd.DataFrame({k: {"duration_min": missions[k].duration_h * 60, "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"], "hours_to_failure": f.result.metrics["hours_to_failure"],
                         "hotspot": tuple(round(x, 1) for x in f.result.metrics["hotspot_location"])} for k, f in fatigue.items()}).T
life

In [ ]:
worst = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst], unit_results["lift"].artifacts["mesh"]))
contrib = pd.Series(fatigue[worst].contributions).sort_values(ascending=False)
ax = contrib.head(8).plot.barh(figsize=(8, 3.2), title=f"{worst}: damage contributions at the hotspot"); ax.invert_yaxis()

In [ ]:
hot = fatigue[worst].hotspot
unit_vm = {k: float(talos.read_frd(r.artifacts["frd"]).von_mises[hot]) / UNIT[k][1] for k, r in unit_results.items()}
rows = {}
for k, sp in spectra.items():
    peak = {}
    for b in sp.blocks:
        peak[b.pattern] = max(peak.get(b.pattern, 0.0), abs(b.mean) + abs(b.amplitude))
    stress = sum(peak.get(pat, 0.0) * unit_vm[pat] for pat in unit_vm)
    rows[k] = {**{f"peak_{pat}": peak.get(pat, 0.0) for pat in unit_vm}, "hotspot_stress_MPa": stress, "SF_yield": LW_PLA.yield_strength / stress}
pd.DataFrame(rows).T.round(2)

## 6. What the numbers say — and the vibration amplitude the resonance really causes

Read the damage column first: at ~10⁻¹⁵ per mission this wing is **not fatigue-limited** — the
solid-equivalent stresses are a fraction of a MPa against an S-N knee of tens of MPa. That is a
result, not a failure of the method: the spectrum, the hotspot and the ranking of the missions are
recorded, and the same pipeline will bite when the wing becomes a 1.2 mm printed shell (the stresses
scale with the section modulus ratio, an input for the next revision).

What the resonance *does* cause is motion: the nacelle oscillates at the shaft frequency with an
amplitude = amplification × unbalance force × static compliance. That is the number to compare with
what the camera and the propeller bearings tolerate. The table gives it per operating point for the
wing as designed, with the resonance avoided (cruise rpm moved), and for the thinner NACA 2412 wing.

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: m.duration_h for k, m in missions.items()}
usage = {"survey": 0.5, "patrol": 0.3, "windy_hops": 0.2}
rate = sum(usage[k] * damage[k] for k in usage) / sum(usage[k] * hours[k] for k in usage)
print(f"damage per 1000 flight hours with usage {usage}: {rate * 1000:.3g}  ->  "
      f"{'not life-limiting (> 1e6 h)' if rate * 1e6 < 1 else f'{1 / rate:.0f} h to failure'}")
sim = chronos.simulate_life(damage, hours, usage, n_flights=2000, seed=0)
fig = sim.plot()

In [ ]:
def assess_design(step, label):
    # a new mesh, modal analysis and unit cases for another wing STEP; same regions, masses, missions, curve
    work = RUNS / label
    m0 = model([], "modal", step); m0.mesh(work / "mesh")
    md_ = m0.solve_modes(work / "mesh", n_modes=8)
    st = chronos.Structure(tuple(md_.metrics["frequencies_hz"]), 0.03)
    units = {}
    for name, (mm, load) in UNIT.items():
        shutil.copytree(work / "mesh", work / name)
        units[name] = (model(mm.loads, name, step).solve(work / name), load)
    dmg = {kk: talos.assess_fatigue(units, chronos.build_spectrum(mission, st).to_dict(), CURVE).result.metrics["damage_per_pass"]
           for kk, mission in missions.items()}
    return st, units, dmg

no_resonance = {kk: talos.assess_fatigue(unit_cases, chronos.build_spectrum(m, None).to_dict(), CURVE).result.metrics["damage_per_pass"]
                for kk, m in missions.items()}
thin_step = drone.generate(part="wing", thickness=0.12).export_step(RUNS / "cad" / "wing_2412.step")
thin_structure, thin_units, thin_damage = assess_design(thin_step, "naca2412")
print("NACA 2412 modes:", [round(x, 1) for x in thin_structure.modes_hz])

def nacelle_amplitude_mm(st, units, point):
    compliance = units["vib_left"][0].metrics["max_displacement"] / UNIT["vib_left"][1]        # mm per N, static
    f = PROP[point]["rpm"] / 60
    daf = float(st.amplification(f)[0]) if st is not None else 1.0
    return daf * PROP[point]["unbalance_N"] * compliance

variants = {"NACA 2415 (as is)": (structure, unit_cases, damage), "NACA 2415, cruise rpm moved off the mode": (None, unit_cases, no_resonance),
            "NACA 2412 (thinner)": (thin_structure, thin_units, thin_damage)}
rows = {}
for name, (st, units, dmg) in variants.items():
    r = {f"nacelle amplitude {pt} [mm]": nacelle_amplitude_mm(st, units, pt) for pt in PROP}
    r["damage per 1000 h (usage mix)"] = sum(usage[kk] * dmg[kk] for kk in usage) / sum(usage[kk] * hours[kk] for kk in usage) * 1000
    rows[name] = r
pd.DataFrame(rows).T

## 7. Export

In [ ]:
summary = {"design": {"file": "designs/fixed_wing.py", "parameters": p}, "modes_hz": freqs, "damping_ratio": 0.03,
           "excitations_hz": lines, "margins": margins.to_dict(),
           "unit_cases": {k: {"load": UNIT[k][1], "max_von_mises": unit_results[k].metrics["max_von_mises"]} for k in UNIT},
           "curve": CURVE.__dict__, "missions": {k: m.describe() for k, m in missions.items()},
           "damage_per_mission": damage, "damage_no_resonance": no_resonance, "damage_naca2412": thin_damage,
           "hours_per_mission": hours, "usage": usage, "damage_per_1000h": rate * 1000, "nacelle_amplitude_mm": rows,
           "spectra": {k: str(RUNS / f"spectrum_{k}.json") for k in missions}}
(RUNS / "life.json").write_text(json.dumps(summary, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Reading the result:** the amplitude column is the design lever — a mode on the cruise shaft
frequency turns a 0.2 N unbalance into millimetres of nacelle motion, and moving the rpm (or the mode)
removes it; the damage column says fatigue is not what limits this wing. Both are recorded with the
missions and curves that produced them. The decision, and the revision that follows, is yours — this
notebook only makes it visible and repeatable.

# Part 2 — the fuselage over a long mission

Part 1 followed the wing. The fuselage has a different job: it **hangs from the wing** (the wing centre section
clamps the cylindrical middle of the fuselage) and carries, as a cantilever forward and aft of that clamp,

- its own weight and the **nose contents** (battery, flight controller, camera) times the load factor *n*,
- the **tail loads**: the balancing download of the horizontal tail and every vertical gust it meets,
- the **fin side loads** from side gusts and turns,
- the **touchdown** (a short 3 g deceleration).

A long mission is where the many small cycles add up: a **full-battery survey** (the endurance from 09a) in
continuous light turbulence. The chain is the same as Part 1 — CAD (Dedalus) → mesh and unit load cases (Talos) →
mission and rainflow spectrum (Chronos) → damage and life (Talos fatigue) — on the **whole fuselage with its tail**,
printed as a 2 mm LW-PLA shell.

In [ ]:
RUNS = ROOT / "fuselage"; RUNS.mkdir()
pf = drone.resolve(part="fuselage", **PREFERRED)
fus = drone.generate(part="fuselage", **PREFERRED)
fcad = fus.export(RUNS / "cad")
mf = fus.measure()
r_f, tt = pf["fuselage_diameter"] / 2, pf["tail_thickness"]
x_nose, x_cyl_end = -pf["nose_length"], -pf["nose_length"] + 0.55 * pf["fuselage_length"]
xt, tc = -pf["nose_length"] + pf["fuselage_length"] - pf["tail_chord"], pf["tail_chord"]
FUS_MASS_KG = mf["volume"] * LW_PLA.density * 1e3                       # t/mm^3 x mm^3 = t -> kg
NOSE_KG = parts.loc[["battery 3S 5000 mAh", "flight controller + GPS", "payload (camera)", "receiver, wiring, bolts"], "mass_g"].sum() / 1000
print(f"fuselage + tail shell: {FUS_MASS_KG * 1000:.0f} g LW-PLA (wall {pf['fuselage_wall']} mm), nose contents {NOSE_KG * 1000:.0f} g, "
      f"valid solid {mf['valid']}")
dviz.show(dviz.plot3d(fus))

## 1. Regions, masses, one mesh

The cylinder under the wing is clamped (the wing centre section and its bolts); the nose carries the battery,
avionics and camera; the horizontal tail and the fin take the tail loads. One mesh serves the modal analysis and
every unit case.

In [ ]:
x_te = min(pf["root_chord"], x_cyl_end - 1.0)                          # the fuselage faces are split at the wing LE and TE
FREGIONS = [
    talos.SurfacesInBox("saddle", (-0.5, -r_f - 1, -2 * r_f - 1, x_te + 0.5, r_f + 1, 1.0)),            # cylinder under the wing
    talos.SurfacesInBox("nose", (x_nose - 1, -r_f - 1, -2 * r_f - 1, 0.5, r_f + 1, 1.0)),                # nose + cylinder ahead of the wing
    talos.SurfacesInBox("hstab", (xt - 0.5, -pf["tail_span"] / 2 - 1, -r_f - tt / 2 - 0.5, xt + tc + 0.5, pf["tail_span"] / 2 + 1, -r_f + tt / 2 + 0.5)),
    talos.SurfacesInBox("fin", (xt - 0.5, -tt / 2 - 0.5, -r_f - 0.5, xt + tc + 0.5, tt / 2 + 0.5, -r_f + pf["fin_height"] + 0.5)),
]
FMASSES = [talos.PointMass("nose", NOSE_KG * 1e-3)]                   # tonnes in mm-N-MPa
FMESH = talos.MeshSettings(element_size=5.0)

def fmodel(loads, name):
    return talos.StructuralModel(fcad.artifacts["step"], "mm-N-MPa", LW_PLA, FREGIONS, [talos.FixedSupport("saddle")],
                                 loads, FMESH, name=name, masses=FMASSES)

fbase = fmodel([], "fuselage_modal")
mres = fbase.mesh(RUNS / "mesh", progress=True)
print(mres)
mres.raise_for_status()                                                 # a region that selects nothing stops here

def fcase_dir(name):
    d_ = RUNS / name
    if not d_.exists():
        shutil.copytree(RUNS / "mesh", d_)
    return d_

## 2. Modes of the fuselage and the propeller lines

The propellers sit on the wing, but their 1P (unbalance) and 2P (blade passing) forces reach the fuselage through
the wing. A tail-boom bending mode close to those lines would shake the tail and the camera.

In [ ]:
fmodes = fbase.solve_modes(fcase_dir("modal"), n_modes=6, progress=True)
ffreqs = fmodes.metrics["frequencies_hz"]
print("fuselage modes [Hz]:", [round(x, 1) for x in ffreqs])
tviz.show(tviz.plot_mode(fmodes, mode=1))
fstructure = chronos.Structure(tuple(ffreqs), damping_ratio=0.03, source="Talos modal, fuselage shell clamped at the wing, nose masses")
fig = fstructure.campbell({"1P": 1, "2P": BLADES}, np.linspace(2000, 10000, 30), operating_rpm={k: v["rpm"] for k, v in PROP.items()})
pd.DataFrame({k: {"1P_hz": v["rpm"] / 60, "2P_hz": BLADES * v["rpm"] / 60, "nearest_mode_hz": fstructure.nearest_mode(v["rpm"] / 60),
                  "margin_1P": fstructure.margin(v["rpm"] / 60), "margin_2P": fstructure.margin(BLADES * v["rpm"] / 60)}
              for k, v in PROP.items()}).T.round(2)

## 3. Unit load cases

Each case is solved once; the mission scales and combines them (linear statics).

| case | load | level in the mission |
|---|---|---|
| `inertia` | 1 g on the shell + the nose contents' weight | load factor *n* |
| `tail_lift` | 5 N up on the horizontal tail | tail force [N] (the trim download is negative) |
| `fin_side` | 3 N sideways on the fin | fin force [N] |

In [ ]:
FUNIT = {
    "inertia":   (fmodel([talos.Acceleration(az=-9810.0), talos.Force("nose", fz=-NOSE_KG * 9.81)], "inertia"), 1.0),
    "tail_lift": (fmodel([talos.Force("hstab", fz=5.0)], "tail_lift"), 5.0),
    "fin_side":  (fmodel([talos.Force("fin", fy=3.0)], "fin_side"), 3.0),
}
funit_results = {}
for name, (m_, load) in tqdm(FUNIT.items(), desc="fuselage unit cases"):
    funit_results[name] = m_.solve(fcase_dir(name))
    r_ = funit_results[name]
    print(f"{name:<10} {'ok' if r_.ok else 'FAILED'}  max von Mises {r_.metrics.get('max_von_mises', float('nan')):.3f} MPa, "
          f"max displacement {r_.metrics.get('max_displacement', float('nan')):.3f} mm at level {load:g}")
funit_cases = {k: (funit_results[k], FUNIT[k][1]) for k in FUNIT}
tviz.show(tviz.plot_results(funit_results["tail_lift"], field="von_mises"))

## 4. The long mission

Gust loads from the sharp-edged gust formula ΔL = ½ ρ V a S u (lift slopes of the flat tail plates and the gust
speeds are **assumptions**): light turbulence 2 m/s at 0.2 gusts per second for the whole survey, side gusts at
0.1 per second, 20 moderate 5 m/s gusts, 12 turns, one touchdown.

In [ ]:
S_T = pf["tail_chord"] * pf["tail_span"] / 1e6                    # m^2 horizontal tail
S_F = pf["tail_chord"] * pf["fin_height"] / 1e6                   # m^2 fin
A_W, A_T, A_F = 2 * math.pi * 0.8, 3.5, 2.5                       # lift slopes [1/rad]: wing (AR ~6), low-AR plates (assumed)
V = V_CRUISE

def dn_gust(u):  return 0.5 * RHO * V * A_W * S * u / W         # wing load-factor increment
def tail_gust(u): return 0.5 * RHO * V * A_T * S_T * u           # N on the horizontal tail
def fin_gust(u):  return 0.5 * RHO * V * A_F * S_F * u           # N on the fin

TAIL_TRIM = -0.03 * W                                             # N: small balancing download (assumed)
T_FLIGHT = 0.9 * endurance_min * 60                               # s: a full battery (09a endurance, 10 % reserve)
U_L, U_M, GUST_S = 2.0, 5.0, 0.5
N_LIGHT, N_SIDE, N_MOD, LEGS = int(0.2 * T_FLIGHT), int(0.1 * T_FLIGHT), 20, 12
T_LEGS = T_FLIGHT - 150 - LEGS * 8 - (N_LIGHT + N_SIDE + N_MOD) * GUST_S

long_mission = chronos.Mission("long_survey", (
    chronos.Segment("take-off + climb", 60, {"inertia": 1.2, "tail_lift": 1.3 * TAIL_TRIM}),
    chronos.Segment("survey legs", T_LEGS, {"inertia": 1.0, "tail_lift": TAIL_TRIM}),
    chronos.Segment("turn between legs", 8, {"inertia": 1.3, "tail_lift": 1.3 * TAIL_TRIM, "fin_side": 0.5 * fin_gust(U_L)}, repeat=LEGS),
    chronos.Segment("light vertical gust", GUST_S, {"inertia": 1 + dn_gust(U_L), "tail_lift": TAIL_TRIM + tail_gust(U_L)}, repeat=N_LIGHT),
    chronos.Segment("side gust", GUST_S, {"inertia": 1.0, "tail_lift": TAIL_TRIM, "fin_side": fin_gust(U_L)}, repeat=N_SIDE),
    chronos.Segment("moderate gust", GUST_S, {"inertia": 1 + dn_gust(U_M), "tail_lift": TAIL_TRIM + tail_gust(U_M), "fin_side": fin_gust(U_M)}, repeat=N_MOD),
    chronos.Segment("descent + landing", 90, {"inertia": 0.9, "tail_lift": TAIL_TRIM}),
    chronos.Segment("touchdown", 1, {"inertia": 3.0}),
), f"full-battery survey ({T_FLIGHT / 60:.0f} min): {LEGS} legs, {N_LIGHT} light and {N_MOD} moderate vertical gusts, {N_SIDE} side gusts, one touchdown")
print(long_mission.description)
print(f"gusts: light dn {dn_gust(U_L):.2f}, moderate dn {dn_gust(U_M):.2f}; tail {tail_gust(U_L):.2f} / {tail_gust(U_M):.2f} N; "
      f"fin {fin_gust(U_L):.2f} / {fin_gust(U_M):.2f} N; tail trim {TAIL_TRIM:.2f} N")
fig = long_mission.profile(patterns=["inertia", "tail_lift", "fin_side"])
fspec = chronos.build_spectrum(long_mission, fstructure)
fspec.save(RUNS / "spectrum_long_survey.json")
fig = fspec.plot()
fspec.table().round(4)

## 5. Damage, hotspot and life

The same **assumed** LW-PLA S-N curve as Part 1 (`CURVE`): coupon tests first before trusting absolute numbers.

In [ ]:
ffat = talos.assess_fatigue(funit_cases, fspec.to_dict(), CURVE, workdir=RUNS / "fatigue")
fm = ffat.result.metrics
flife = pd.Series({"mission_min": long_mission.duration_h * 60, "damage_per_mission": fm["damage_per_pass"],
                   "missions_to_failure": fm["passes_to_failure"], "hours_to_failure": fm["hours_to_failure"],
                   "hotspot_mm": tuple(round(x, 1) for x in fm["hotspot_location"])})
print(flife.to_string())
tviz.show(tviz.plot_damage(ffat, funit_results["inertia"].artifacts["mesh"]))
fcontrib = pd.Series(ffat.contributions).sort_values(ascending=False)
ax = fcontrib.head(8).plot.barh(figsize=(8, 3.2), title="long survey: damage contributions at the fuselage hotspot"); ax.invert_yaxis()

In [ ]:
# static re-check: the largest level of each pattern in the mission, combined at the hotspot and at each case's own peak
fhot = ffat.hotspot
f_unit_vm = {k: float(talos.read_frd(r_.artifacts["frd"]).von_mises[fhot]) / FUNIT[k][1] for k, r_ in funit_results.items()}
fpeak = {}
for b in fspec.blocks:
    fpeak[b.pattern] = max(fpeak.get(b.pattern, 0.0), abs(b.mean) + abs(b.amplitude))
hot_stress = sum(fpeak.get(k, 0.0) * f_unit_vm[k] for k in f_unit_vm)
bound = sum(fpeak.get(k, 0.0) * funit_results[k].metrics["max_von_mises"] / FUNIT[k][1] for k in FUNIT)   # peaks added: conservative
print(f"peak levels {({k: round(v, 2) for k, v in fpeak.items()})}")
print(f"hotspot stress at the peaks {hot_stress:.2f} MPa -> SF on yield {LW_PLA.yield_strength / hot_stress:.1f}; "
      f"conservative bound (every case at its own peak) {bound:.2f} MPa -> SF {LW_PLA.yield_strength / bound:.1f}")
sim_f = chronos.simulate_life({"long_survey": fm["damage_per_pass"]}, {"long_survey": long_mission.duration_h}, {"long_survey": 1.0},
                              n_flights=2000, seed=0)
fig = sim_f.plot()

In [ ]:
fuselage_summary = {
    "design": {"file": "designs/fixed_wing.py", "part": "fuselage", "parameters": pf},
    "mass_kg": {"shell": FUS_MASS_KG, "nose_contents": NOSE_KG}, "modes_hz": ffreqs,
    "mission": long_mission.describe(), "gusts": {"light_m_s": U_L, "moderate_m_s": U_M, "slopes_per_rad": {"wing": A_W, "tail": A_T, "fin": A_F}},
    "fatigue": {k: v for k, v in fm.items() if not isinstance(v, (list, dict))} | {"hotspot_location": fm["hotspot_location"]},
    "static": {"hotspot_stress_MPa": hot_stress, "bound_MPa": bound, "yield_MPa": LW_PLA.yield_strength},
    "assumptions": ["LW-PLA S-N curve assumed", "flat-plate tail lift slopes", "sharp-edged gusts", "wing clamps the whole fuselage cylinder"],
}
(RUNS / "fuselage_life.json").write_text(json.dumps(fuselage_summary, indent=2, default=str))
print("written", RUNS / "fuselage_life.json")

**Reading the result:** the fuselage is a cantilever on each side of the wing — the tail cone behind it carries the
tail and fin gusts, the nose ahead of it the battery and camera. The hotspot and its contributions say which one
limits the shell; the Campbell table says whether a tail-boom mode sits on a propeller line. The levers are the
wall thickness (`fuselage_wall`), the tail-cone length and a stiffer print in the cone.